In [ ]:
import pandas as pd
import numpy as np


In [19]:
ENDERECO_DADOS = 'https://www.ispdados.rj.gov.br/Arquivos/BaseDPEvolucaoMensalCisp.csv'

df_roubos = pd.read_csv(ENDERECO_DADOS, sep=';' , encoding= 'iso-8859-1')
# print(df_roubos.head())
# print(df_roubos.describe())
# print (df_roubos['regiao'].unique)




In [20]:
# Delimitando as variáveis cisp regiao munic roubo_veiculo
df_roubos = df_roubos[['cisp','regiao', 'munic', 'roubo_veiculo']]
# Corrigindo a Grande Niterói
correcao_acentuacao = df_roubos['regiao'].str.startswith('Grande Niter', na=False)
# print(correcao_acentuacao)
df_roubos.loc[correcao_acentuacao, 'regiao'] = 'Grande Niterói'
print(df_roubos['regiao'].unique())


<StringArray>
['Capital', 'Baixada Fluminense', 'Interior', 'Grande Niterói']
Length: 4, dtype: str


In [21]:
# Agrupando e totalizando os roubos de veículo por cisp regiao e munic
df_roubos = df_roubos.groupby(['cisp', 'regiao', 'munic'], as_index=False)['roubo_veiculo'].sum()
print(df_roubos.head(30))

    cisp   regiao           munic  roubo_veiculo
0      1  Capital  Rio de Janeiro            583
1      4  Capital  Rio de Janeiro           2230
2      5  Capital  Rio de Janeiro           1384
3      6  Capital  Rio de Janeiro           4708
4      7  Capital  Rio de Janeiro           2035
5      9  Capital  Rio de Janeiro           4701
6     10  Capital  Rio de Janeiro           2957
7     11  Capital  Rio de Janeiro             18
8     12  Capital  Rio de Janeiro            323
9     13  Capital  Rio de Janeiro            315
10    14  Capital  Rio de Janeiro           1063
11    15  Capital  Rio de Janeiro           1268
12    16  Capital  Rio de Janeiro           5948
13    17  Capital  Rio de Janeiro           7771
14    18  Capital  Rio de Janeiro           7393
15    19  Capital  Rio de Janeiro           6953
16    20  Capital  Rio de Janeiro           9201
17    21  Capital  Rio de Janeiro          19884
18    22  Capital  Rio de Janeiro          15236
19    23  Capital  R

In [34]:
# Calculando os Quartis
array_roubos = np.array(df_roubos['roubo_veiculo'])

media = np.mean(array_roubos)
mediana = np.median(array_roubos)

maximo = np.max(array_roubos)
minimo = np.min(array_roubos)
total = np.sum(array_roubos)

q1 = np.percentile(array_roubos,25)
q3 = np.percentile(array_roubos,75)

print(f'Média: {media}')
print(f'Mediana: {mediana}')

print(f'Máximo: {maximo:.2f}')
print(f'Mínimo:  {minimo:.2f}')
print(f'Total:  {total:.2f}')

print( f'Q1: {q1:.2f}' )
print( f'Q3: {q3:.2f}' )






Média: 4969.843537414966
Mediana: 942.0
Máximo: 38084.00
Mínimo:  2.00
Total:  730567.00
Q1: 90.00
Q3: 7390.00


In [38]:
# Criando uma série com os maiores
df_maiores = df_roubos[df_roubos['roubo_veiculo'] >=q3].copy()
df_maiores['flag'] = 'mais'
display(df_maiores)

df_menores = df_roubos[df_roubos['roubo_veiculo'] <=q1].copy()
df_menores['flag'] = 'menos'
display(df_menores)

,cisp,regiao,munic,roubo_veiculo,flag
13,17,Capital,Rio de Janeiro,7771,mais
14,18,Capital,Rio de Janeiro,7393,mais
16,20,Capital,Rio de Janeiro,9201,mais
17,21,Capital,Rio de Janeiro,19884,mais
18,22,Capital,Rio de Janeiro,15236,mais
19,23,Capital,Rio de Janeiro,10833,mais
20,24,Capital,Rio de Janeiro,12311,mais
21,25,Capital,Rio de Janeiro,11140,mais
22,26,Capital,Rio de Janeiro,7920,mais
23,27,Capital,Rio de Janeiro,28171,mais


,cisp,regiao,munic,roubo_veiculo,flag
7,11,Capital,Rio de Janeiro,18,menos
41,45,Capital,Rio de Janeiro,27,menos
79,91,Interior,Valença,49,menos
80,92,Interior,Rio das Flores,8,menos
83,94,Interior,Piraí;Pinheiral,53,menos
85,96,Interior,Miguel Pereira;Paty dos Alferes,43,menos
86,97,Interior,Mendes,23,menos
87,98,Interior,Engenheiro Paulo de Frontin,18,menos
89,100,Interior,Porto Real;Quatis,87,menos
90,101,Interior,Pinheiral,47,menos


In [36]:
# Concatenando os dois dataframes df_maiores e df_menores 
df_roubos_flags = pd.concat([df_maiores,df_menores], ignore_index=True)


In [37]:
# Exportando os dados utf-8-sig p/ ler o powerbi
df_roubos_flags.to_csv('roubos_flag.csv', index=False, encoding='utf-8-sig')